In [10]:
from datasets import load_dataset
from IPython.display import Audio, display
import numpy as np
import soundfile as sf

ds = load_dataset("minhtien2405/fptu-vovinam-dataset")
sample = ds["train"][0]

# Trường hợp đã có torchcodec (decode=True mặc định)
try:
    audio_obj = sample["audio"]  # AudioDecoder
    decoded = audio_obj.get_all_samples()
    wav = decoded.data
    if hasattr(wav, "cpu"):  # torch tensor
        wav = wav.cpu().numpy()
    wav = np.asarray(wav).squeeze()
    sr = decoded.sample_rate
except Exception:
    # Fallback nếu chưa decode được: đọc trực tiếp từ file path
    from datasets import Audio as HFAudio
    ds2 = ds.cast_column("audio", HFAudio(decode=False))
    sample = ds2["train"][0]
    wav, sr = sf.read(sample["audio"]["path"])
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

display(Audio(wav, rate=sr))
print(sample.get("text", "No text field"))


truyền bá võ thuật cho quần chúng


In [12]:
from datasets import load_dataset, concatenate_datasets
from collections import Counter

# Load dataset
ds = load_dataset("minhtien2405/fptu-vovinam-dataset")

REGION_COL = "region"  # đổi nếu dataset bạn dùng tên cột khác

# 1) Check region theo từng split
print("=== Region theo từng split ===")
for split_name, split_ds in ds.items():
    if REGION_COL not in split_ds.column_names:
        print(f"[{split_name}] Không có cột '{REGION_COL}'. Columns: {split_ds.column_names}")
        continue

    regions = [
        str(x).strip() if x is not None and str(x).strip() != "" else "MISSING"
        for x in split_ds[REGION_COL]
    ]
    counts = Counter(regions)

    print(f"\n[{split_name}]")
    for region, n in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
        print(f"  {region}: {n}")

# 2) Check toàn bộ dataset (gộp tất cả split)
print("\n=== Region toàn bộ dataset ===")
all_ds = concatenate_datasets([split_ds for split_ds in ds.values() if REGION_COL in split_ds.column_names])
all_regions = [
    str(x).strip() if x is not None and str(x).strip() != "" else "MISSING"
    for x in all_ds[REGION_COL]
]
all_counts = Counter(all_regions)

for region, n in sorted(all_counts.items(), key=lambda x: (-x[1], x[0])):
    print(f"{region}: {n}")

print("\nUnique regions:", sorted(all_counts.keys()))


=== Region theo từng split ===

[train]
  Central: 13336
  North: 602
  South: 110

[validation]
  Central: 1682
  North: 59
  South: 15

[test]
  Central: 1693
  North: 60
  South: 3

=== Region toàn bộ dataset ===
Central: 16711
North: 721
South: 128

Unique regions: ['Central', 'North', 'South']
